# FDS Challenge: My Notebook

### 0. Configuration

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from lightgbm import LGBMClassifier

    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False

try:
    from IPython.display import display
except ImportError:
    display = print

DEBUG = True
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (cwd, cwd.parent) if (candidate / "dataset").is_dir()),
    cwd,
)
DATASET_DIR = PROJECT_ROOT / "dataset"
TRAIN_FILE_PATH = str(DATASET_DIR / "train.jsonl")
TEST_FILE_PATH = str(DATASET_DIR / "test.jsonl")
SAMPLE_SUBMISSION_PATH = str(DATASET_DIR / "sample_submission.csv")

# Available models: "ensemble", "lightgbm", "random_forest", "logistic_regression"
"""
logistic_regression:
A simple linear model that weights each feature and outputs a win probability.
It is fast and easy to interpret, but it only captures straightforward relationships.
Features are standardized first.

random_forest:
Trains 150 decision trees on different random subsets of the data and averages their votes.
It handles non-linear patterns well and is robust to noise and overfitting.

lightgbm:
A gradient boosting model that builds trees one after another, each correcting the errors of the previous ones.
It is usually the most accurate on tabular data.

ensemble:
A soft-voting combination of the three models above, averaging their probabilities with weights of 50% LightGBM, 30% Random Forest and 20% Logistic Regression.
It tends to be the most stable, but it is the slowest because it trains three models per fold.
"""
MODEL_TYPE = "ensemble"
K_FOLDS = 5
RANDOM_STATE = 42


def print_battle(data, battle_idx=0):
    """
    Prints the whole battle at index `battle_idx`
    """

    if not data:
        return

    battle = data[battle_idx]
    output_path = f"battle_{battle_idx}.json"
    with open(output_path, "w") as f:
        f.write(json.dumps(battle, indent=4))

    print(f"SUCCESS: battle saved to {output_path}.")


### 1. Loading the Data


In [ ]:
def load_data(file_path):
    """
    Loads the whole `file_path` and stores it as a list of battles
    """

    print(f"Loading data from '{file_path}'...")
    data = []
    with open(file_path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    print(f"SUCCESS: loaded {len(data)} battles.")
    if DEBUG:
        print_battle(data)

    return data


# Loading data
train_data = load_data(TRAIN_FILE_PATH)
test_data = load_data(TEST_FILE_PATH)

In [ ]:
TYPE_CHART_PATH = str(DATASET_DIR / "type_chart_gen2.csv")
TYPE_MAP_PATH = str(PROJECT_ROOT / "type_map.json")

def normalize_type_map(type_map):
    return {name.strip().lower(): [pokemon_type.strip().lower() for pokemon_type in types] for name, types in type_map.items()}

def normalize_type_chart_csv(type_chart):
    type_chart = type_chart.copy()
    type_chart.columns = type_chart.columns.str.strip()
    type_chart["attacking_type"] = type_chart["attacking_type"].str.strip().str.lower()
    type_chart["defending_type"] = type_chart["defending_type"].str.strip().str.lower()
    return type_chart

def load_type_map(file_path):
    try:
        with open(file_path, "r") as f:
            return normalize_type_map(json.load(f))
    except FileNotFoundError:
        return {}

def load_type_chart(file_path):
    try:
        return normalize_type_chart_csv(pd.read_csv(file_path))
    except FileNotFoundError:
        print(f"ERROR: Could not find the file at '{file_path}'.")
        return pd.DataFrame()

def create_type_map(data, file_path=TYPE_MAP_PATH):
    type_map = load_type_map(file_path)
    if type_map:
        return type_map

    type_map = {}
    for battle in data:
        for pokemon in battle.get("p1_team_details", []):
            if pokemon.get("name") and pokemon.get("types"):
                type_map[pokemon["name"]] = pokemon["types"]
        lead = battle.get("p2_lead_details", {})
        if lead.get("name") and lead.get("types"):
            type_map[lead["name"]] = lead["types"]

    type_map = normalize_type_map(type_map)
    with open(file_path, "w") as f:
        json.dump(type_map, f, indent=4)
    return type_map

type_map = create_type_map(train_data)
type_chart = load_type_chart(TYPE_CHART_PATH)

### 2. Extract Features (both Train and Test Features)

Now we are only using the number of dead pokemons (identified by status = "fnt") as feature. The assumption is that "more dead pokemons => less likelihood to win"


In [ ]:
def add_fnt_count(features, turn):
    p1_state = turn["p1_pokemon_state"]
    p2_state = turn["p2_pokemon_state"]

    # Estraiamo le feature riguardanti i pokemon esausti
    if p1_state["status"] == "fnt": features["p1_fnt"] = features.get("p1_fnt", 0) + 1
    if p2_state["status"] == "fnt": features["p2_fnt"] = features.get("p2_fnt", 0) + 1

def get_hp_pct_lost(battle: dict, player: int) -> list:
    # Danno subito da player[player] nei turni 0-10, 10-20 e 20-30
    dmg_diffs = []
    # Seleziona il player corretto
    player_key = "p1_pokemon_state" if player == 1 else "p2_pokemon_state"

    # Dizionario della squadra:
    # name: {lst_hp, diff_0_10, diff_10_20, diff_20_30}
    # lst_hp := the last seen hp of the pokemon
    # diff_a_b := the average of percentage hp lost across all the team between turn a and b
    team = {}

    for i, turn in enumerate(battle["battle_timeline"]):
        # Ottieni pokemon e hp del player nel turno corrente
        pkm_name = turn.get(player_key, {}).get("name")
        pkm_hp = turn.get(player_key, {}).get("hp_pct", 1.0)

        # Se è già comparso ottieni l'ultima occorrenza di hp
        if pkm_name in team:
            pkm_lst_hp = team.get(pkm_name).get("lst_hp")
        # Altrimenti inizializza
        else:
            team[pkm_name] = {"lst_hp": 1, "diff_0_10": 0, "diff_10_20": 0, "diff_20_30": 0}
            pkm_lst_hp = 1

        # Aggiungi la differenza di hp al pokemon corrente
        diff = pkm_lst_hp - pkm_hp
        match (i // 10 + 1):
            case 1:
                team.get(pkm_name)["diff_0_10"] += diff
            case 2:
                team.get(pkm_name)["diff_10_20"] += diff
            case 3:
                team.get(pkm_name)["diff_20_30"] += diff

        # Ai turni 10, 20 e 30 calcola la media degli hp persi
        match (i):
            case 9:
                mean1 = sum([pkm["diff_0_10"] for pkm in team.values()]) / len(team.keys())
                dmg_diffs.append(mean1)
            case 19:
                mean2 = sum([pkm["diff_10_20"] for pkm in team.values()]) / len(team.keys())
                dmg_diffs.append(mean2)
            case 29:
                mean3 = sum([pkm["diff_20_30"] for pkm in team.values()]) / len(team.keys())
                dmg_diffs.append(mean3)
        # Aggiorna l'hp per la prossima occorrenza del pokemon
        team.get(pkm_name)["lst_hp"] = pkm_hp

    return (dmg_diffs + [0.0, 0.0, 0.0])[:3]


def add_multiplier_counts(features, turn, att_player, def_player, type_map, type_chart):
    att_move = turn[f"{att_player}_move_details"]
    def_pkm = turn[f"{def_player}_pokemon_state"]

    # cold happen that att_move == null if the player changes the pokemon
    if not att_move or not att_move.get("type"):
        return

    att_type = att_move["type"].lower()
    def_type = type_map.get(def_pkm["name"].strip().lower(), ["unknown"])

    multiplier = get_multiplier(att_type, def_type, type_chart)
    if multiplier == 0:     key = "zero_eff_moves"
    elif multiplier < 1:    key = "low_eff_moves"
    elif multiplier == 1:   key = "normal_eff_moves"
    else:                   key = "super_eff_moves"
    features[f"{att_player}_{key}"] = features.get(f"{att_player}_{key}", 0) + 1
    
def get_multiplier(att_type: str, def_types: list[str], type_chart: pd.DataFrame) -> float:

    # If the defending type is unknown, we assume a neutral multiplier of 1.0
    if def_types[0] == "unknown": return 1.0 
    
    multiplier = 1.0
    for def_type in def_types:
        match = type_chart[(type_chart['attacking_type'] == att_type) & (type_chart['defending_type'] == def_type)]
        if not match.empty: multiplier *= match['multiplier'].iloc[0]

    return multiplier


def extract_features(data, type_map, type_chart):
    """
    Estrae le feature combinando:
    1. Statistiche base del team P1 e del Lead P2 (da starter-notebook.ipynb ed estese).
    2. Dinamica dei 30 turni osservabili (timeline):
       - Tracciamento accurato dei Pokémon esausti (senza duplicazioni).
       - Stima della vita residua totale di squadra (considerando i Pokémon di P2 non ancora scesi in campo).
       - Stato della battaglia al turno 30 (HP attivo, alterazioni di stato maggiori, boost attivi).
       - Numero di switch effettuati.
    """
    feature_list = []
    bad_statuses = {"slp", "frz", "tox", "psn", "par", "brn"}

    for battle in data:
        features = {}

        # =========================================================================
        # 1. Team Base Stats (Starter-notebook & Estensioni)
        # =========================================================================
        p1_team = battle.get("p1_team_details", [])
        if p1_team:
            features["p1_mean_hp"] = np.mean([p.get("base_hp", 0) for p in p1_team])
            features["p1_mean_spe"] = np.mean([p.get("base_spe", 0) for p in p1_team])
            features["p1_mean_atk"] = np.mean([p.get("base_atk", 0) for p in p1_team])
            features["p1_mean_def"] = np.mean([p.get("base_def", 0) for p in p1_team])
            features["p1_mean_spa"] = np.mean([p.get("base_spa", 0) for p in p1_team])
            features["p1_mean_spd"] = np.mean([p.get("base_spd", 0) for p in p1_team])
            features["p1_mean_bst"] = features["p1_mean_hp"] + features["p1_mean_atk"] + features["p1_mean_def"] + features["p1_mean_spa"] + features["p1_mean_spd"] + features["p1_mean_spe"]

        p2_lead = battle.get("p2_lead_details", {})
        if p2_lead:
            features["p2_lead_hp"] = p2_lead.get("base_hp", 0)
            features["p2_lead_spe"] = p2_lead.get("base_spe", 0)
            features["p2_lead_atk"] = p2_lead.get("base_atk", 0)
            features["p2_lead_def"] = p2_lead.get("base_def", 0)
            features["p2_lead_spa"] = p2_lead.get("base_spa", 0)
            features["p2_lead_spd"] = p2_lead.get("base_spd", 0)
            features["p2_lead_bst"] = features["p2_lead_hp"] + features["p2_lead_atk"] + features["p2_lead_def"] + features["p2_lead_spa"] + features["p2_lead_spd"] + features["p2_lead_spe"]

        # Differenziali di statistiche Team P1 vs Lead P2
        if p1_team and p2_lead:
            features["diff_mean_hp"] = features["p1_mean_hp"] - features["p2_lead_hp"]
            features["diff_mean_spe"] = features["p1_mean_spe"] - features["p2_lead_spe"]
            features["diff_mean_atk"] = features["p1_mean_atk"] - features["p2_lead_atk"]
            features["diff_mean_def"] = features["p1_mean_def"] - features["p2_lead_def"]
            features["diff_mean_bst"] = features["p1_mean_bst"] - features["p2_lead_bst"]

        # =========================================================================
        # 2. Battle Timeline Dynamics (Primi 30 turni)
        # =========================================================================
        timeline = battle.get("battle_timeline", [])

        # Mappatura HP e stato per ciascun Pokémon
        p1_hp = {p["name"]: 1.0 for p in p1_team} if p1_team else {}
        p2_hp = {p2_lead["name"]: 1.0} if p2_lead and "name" in p2_lead else {}

        p1_status_map = {}
        p2_status_map = {}

        p1_switches = 0
        p2_switches = 0
        prev_p1_poke = None
        prev_p2_poke = None

        for turn in timeline:
            add_multiplier_counts(features, turn, "p1", "p2", type_map, type_chart)
            add_multiplier_counts(features, turn, "p2", "p1", type_map, type_chart)

            p1_state = turn.get("p1_pokemon_state", {})
            p2_state = turn.get("p2_pokemon_state", {})

            p1_name = p1_state.get("name")
            p2_name = p2_state.get("name")

            if p1_name:
                p1_hp[p1_name] = 0.0 if p1_state.get("status") == "fnt" else p1_state.get("hp_pct", 1.0)
                p1_status_map[p1_name] = p1_state.get("status", "nostatus")
                if prev_p1_poke and p1_name != prev_p1_poke:
                    p1_switches += 1
                prev_p1_poke = p1_name

            if p2_name:
                p2_hp[p2_name] = 0.0 if p2_state.get("status") == "fnt" else p2_state.get("hp_pct", 1.0)
                p2_status_map[p2_name] = p2_state.get("status", "nostatus")
                if prev_p2_poke and p2_name != prev_p2_poke:
                    p2_switches += 1
                prev_p2_poke = p2_name

        # Conteggio Pokémon esausti e superstiti (tracciamento univoco)
        p1_fnt = sum(1 for hp in p1_hp.values() if hp == 0.0)
        p2_fnt = sum(1 for hp in p2_hp.values() if hp == 0.0)
        features["p1_fnt"] = p1_fnt
        features["p2_fnt"] = p2_fnt
        features["fnt_diff"] = p2_fnt - p1_fnt

        p1_alive = 6 - p1_fnt
        p2_alive = 6 - p2_fnt
        features["p1_alive"] = p1_alive
        features["p2_alive"] = p2_alive
        features["alive_diff"] = p1_alive - p2_alive

        # Rivelazione squadra P2 (i non visti hanno 100% HP residuo)
        p2_revealed_count = len(p2_hp)
        p2_unseen_count = max(0, 6 - p2_revealed_count)
        features["p2_revealed_count"] = p2_revealed_count
        features["p2_unseen_count"] = p2_unseen_count

        # Hp percentuale perso da P1 e P2 nei turni 0-10, 10-20 e 20-30
        p1_hp_lost = get_hp_pct_lost(battle, 1)
        features["p1_hp_pct_lost_0_10"] = p1_hp_lost[0]
        features["p1_hp_pct_lost_10_20"] = p1_hp_lost[1]
        features["p1_hp_pct_lost_20_30"] = p1_hp_lost[2]

        p2_hp_lost = get_hp_pct_lost(battle, 2)
        features["p2_hp_pct_lost_0_10"] = p2_hp_lost[0]
        features["p2_hp_pct_lost_10_20"] = p2_hp_lost[1]
        features["p2_hp_pct_lost_20_30"] = p2_hp_lost[2]

        # Stato al termine dell'osservazione (Turno 30 o ultimo disponibile)
        if timeline:
            last_turn = timeline[-1]
            last_p1 = last_turn.get("p1_pokemon_state", {})
            last_p2 = last_turn.get("p2_pokemon_state", {})

            features["p1_active_hp"] = last_p1.get("hp_pct", 0.0)
            features["p2_active_hp"] = last_p2.get("hp_pct", 0.0)
            features["active_hp_diff"] = features["p1_active_hp"] - features["p2_active_hp"]

            b1 = last_p1.get("boosts", {})
            b2 = last_p2.get("boosts", {})
            features["p1_boosts_sum"] = sum(b1.values()) if b1 else 0
            features["p2_boosts_sum"] = sum(b2.values()) if b2 else 0
            features["boosts_diff"] = features["p1_boosts_sum"] - features["p2_boosts_sum"]

            features["p1_active_is_fnt"] = 1 if last_p1.get("status") == "fnt" else 0
            features["p2_active_is_fnt"] = 1 if last_p2.get("status") == "fnt" else 0

        # Alterazioni di stato maggiori attive sui Pokémon visti
        features["p1_bad_status_count"] = sum(1 for s in p1_status_map.values() if s in bad_statuses)
        features["p2_bad_status_count"] = sum(1 for s in p2_status_map.values() if s in bad_statuses)
        features["bad_status_diff"] = features["p2_bad_status_count"] - features["p1_bad_status_count"]

        # Mobilità e switch
        features["p1_switches"] = p1_switches
        features["p2_switches"] = p2_switches
        features["switches_diff"] = p1_switches - p2_switches

        # Identificatori e Target
        if "player_won" in battle:
            features["player_won"] = int(battle["player_won"])
        if "battle_id" in battle:
            features["battle_id"] = battle["battle_id"]

        feature_list.append(features)

    return pd.DataFrame(feature_list).fillna(0)


# 2. Estrazione Feature Arricchite
print("\nEstrazione feature in corso...")
train_df = extract_features(train_data, type_map, type_chart)
test_df = extract_features(test_data, type_map, type_chart)

print(f"Dimensioni Training Features: {train_df.shape}")
# print(f"Dimensioni Test Features    : {test_df.shape}")
display(train_df.head())

# Perfomance Metrics

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }

### 3. Training a Baseline Model

Now that we have some features, let's train a simple `LogisticRegression` model. This will give us a starting point for our predictions.

In [ ]:
def build_model(model_type=MODEL_TYPE, random_state=RANDOM_STATE):
    """
    Restituisce un'istanza fresca del modello specificato.
    """
    if model_type == "lightgbm":
        if not LIGHTGBM_AVAILABLE:
            raise ImportError("LightGBM non è installato. Esegui 'pip install lightgbm'.")
        return LGBMClassifier(
            n_estimators=150,
            max_depth=5,
            learning_rate=0.04,
            subsample=0.8,
            colsample_bytree=0.8,
            verbose=-1,
            random_state=random_state,
        )

    elif model_type == "random_forest":
        return RandomForestClassifier(
            n_estimators=150,
            max_depth=8,
            min_samples_split=5,
            random_state=random_state,
            n_jobs=-1,
        )

    elif model_type == "logistic_regression":
        # È fondamentale standardizzare le feature per la convergenza della Logistic Regression
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=1000, C=1.0, random_state=random_state),
        )

    elif model_type == "ensemble":
        if not LIGHTGBM_AVAILABLE:
            raise ImportError("LightGBM non è installato per l'ensemble.")
        clf_lgb = LGBMClassifier(
            n_estimators=150,
            max_depth=5,
            learning_rate=0.04,
            subsample=0.8,
            colsample_bytree=0.8,
            verbose=-1,
            random_state=random_state,
        )
        clf_rf = RandomForestClassifier(
            n_estimators=150,
            max_depth=8,
            random_state=random_state,
            n_jobs=-1,
        )
        clf_lr = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=1000, C=1.0, random_state=random_state),
        )
        return VotingClassifier(
            estimators=[("lgb", clf_lgb), ("rf", clf_rf), ("lr", clf_lr)],
            voting="soft",
            weights=[0.5, 0.3, 0.2],
        )

    else:
        raise ValueError(f"Modello non supportato: {model_type}")


def train_and_evaluate_cv(X, y, X_test, model_type=MODEL_TYPE, k=K_FOLDS, random_state=RANDOM_STATE):
    """
    Esegue Stratified K-Fold Cross-Validation:
    1. Calcola e registra le metriche su ciascun fold.
    2. Calcola le predizioni Out-Of-Fold (OOF) per una stima globale non-biased.
    3. Calcola la predizione media sul test set (K-Fold test averaging / ensembling).
    """
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=random_state)

    oof_probs = np.zeros(len(X))
    test_probs = np.zeros(len(X_test))
    fold_metrics_list = []

    print(f"\n{'='*60}")
    print(f"Avvio Stratified {k}-Fold Cross Validation con modello: {model_type.upper()}")
    print(f"{'='*60}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = build_model(model_type, random_state=random_state + fold)
        model.fit(X_train, y_train)

        # Predizione sul fold di validazione
        val_prob = model.predict_proba(X_val)[:, 1]
        val_pred = (val_prob >= 0.5).astype(int)
        oof_probs[val_idx] = val_prob

        metrics = compute_metrics(y_val, val_pred, val_prob)
        fold_metrics_list.append(metrics)

        print(f"Fold {fold+1}/{k} -> Accuracy: {metrics['accuracy']:.4f} | " f"F1: {metrics['f1']:.4f} | ROC-AUC: {metrics['roc_auc']:.4f}")

        # Accumula predizione sul test set
        test_probs += model.predict_proba(X_test)[:, 1] / k
    # Statistiche aggregate sui K fold
    mean_acc = np.mean([m["accuracy"] for m in fold_metrics_list])
    std_acc = np.std([m["accuracy"] for m in fold_metrics_list])
    mean_f1 = np.mean([m["f1"] for m in fold_metrics_list])
    mean_auc = np.mean([m["roc_auc"] for m in fold_metrics_list])

    # Performance Out-Of-Fold complessiva
    oof_pred = (oof_probs >= 0.5).astype(int)
    overall_oof_acc = accuracy_score(y, oof_pred)
    overall_oof_auc = roc_auc_score(y, oof_probs)

    print(f"\n{'-'*60}")
    print(f"RISULTATI CV AGGREGATI ({k} FOLDS):")
    print(f"Accuracy Media : {mean_acc:.4f} (+/- {std_acc:.4f})")
    print(f"F1 Media       : {mean_f1:.4f}")
    print(f"ROC-AUC Media  : {mean_auc:.4f}")
    print(f"OOF Accuracy   : {overall_oof_acc:.4f}")
    print(f"OOF ROC-AUC    : {overall_oof_auc:.4f}")
    print(f"{'-'*60}\n")

    test_predictions = (test_probs >= 0.5).astype(int)
    return test_predictions, test_probs


# Selezione delle feature escludendo ID e target
feature_cols = [c for c in train_df.columns if c not in ["battle_id", "player_won"]]
print(f"\nNumero di feature utilizzate: {len(feature_cols)}")
print(f"Elenco feature: {feature_cols}")

X_train = train_df[feature_cols]
y_train = train_df["player_won"]
X_test = test_df[feature_cols]

# 3. Addestramento, Validazione K-Fold e Predizione Ensembled
test_predictions, test_probabilities = train_and_evaluate_cv(
    X=X_train,
    y=y_train,
    X_test=X_test,
    model_type=MODEL_TYPE,
    k=K_FOLDS,
    random_state=RANDOM_STATE,
)

### 4. Creating the Submission File

The competition requires a `.csv` file with two columns: `battle_id` and `player_won`. Let's use our trained model to make predictions on the test set and format them correctly.

In [ ]:
import pandas as pd


def predict_and_save(predictions, test_battle_ids, output_path="submission.csv"):
    """
    Formatta e salva il file di sottomissione secondo il template richiesto.
    """
    print(f"Salvataggio delle predizioni in '{output_path}'...")
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    expected_columns = ["battle_id", "player_won"]
    if list(sample_submission.columns) != expected_columns:
        raise ValueError(f"Expected submission columns {expected_columns}, got {list(sample_submission.columns)}")

    predictions_by_id = pd.Series(predictions, index=test_battle_ids)
    submission_df = sample_submission.copy()
    submission_df["player_won"] = submission_df["battle_id"].map(predictions_by_id)

    if submission_df["player_won"].isna().any():
        raise ValueError("Alcuni battle_id nella sample submission non hanno una predizione corrispondente.")

    submission_df["player_won"] = submission_df["player_won"].astype(int)
    submission_df.to_csv(output_path, index=False)

    print(f"File '{output_path}' creato con successo!")
    print(submission_df.head(10))
    print(f"Distribuzione predizioni (0 vs 1): {dict(submission_df['player_won'].value_counts())}")


# 4. Creazione del file di sottomissione
predict_and_save(test_predictions, test_battle_ids=test_df["battle_id"], output_path="submission.csv")